# upsert

Generic  — executes a MERGE INTO on a Delta table by primary key(s).

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import DataFrame


def upsert_delta(
    df: DataFrame,
    target_table: str,
    merge_keys: list,
) -> None:
    """MERGE df into target_table on merge_keys — update matched rows, insert new ones."""
    condition = " AND ".join(f"tgt.{k} = src.{k}" for k in merge_keys)
    (
        DeltaTable.forName(spark, target_table)
        .alias("tgt")
        .merge(df.alias("src"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )